# Build the pooled text emotion dataset (Colab)

Downloads the real EN/RU text emotion corpora, maps them to the 7 MELD emotions, removes duplicates and leaks, and writes train/dev/test files plus `stats.md`.

No GPU needed (CPU runtime is fine). Outputs go to `MyDrive/diploma-mer/data/`.

Run the pilot first (2,000 train rows per source), check `stats.md`, then the full build.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/diploma-mer'
import os
os.makedirs(f'{DRIVE_DIR}/data/raw', exist_ok=True)

In [ ]:
# Clone (or update) the repo
REPO = 'NurimanB/multimodal-emotion-recognition'
BRANCH = 'text-data-pooling'

from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
REPO_DIR = '/content/' + REPO.split('/')[1]

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} https://{token}@github.com/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
!pip install -q pandas pyarrow pyyaml ftfy tabulate
!pip install -q -e . --no-deps

## 1. Pilot
At most 2,000 train rows per source. Raw downloads are cached on Drive (`data/raw`), so the full build reuses them.

In [ ]:
!python -m mer.data.build_text --config configs/data_text_pilot.yaml cache_dir={DRIVE_DIR}/data/raw out_dir={DRIVE_DIR}/data/text_pilot

## 2. Full build

In [ ]:
!python -m mer.data.build_text --config configs/data_text_v1.yaml cache_dir={DRIVE_DIR}/data/raw out_dir={DRIVE_DIR}/data/text_v1

In [ ]:
import pandas as pd
train = pd.read_parquet(f'{DRIVE_DIR}/data/text_v1/train.parquet')
print(len(train))
train.sample(10, random_state=0)[['source', 'lang', 'emotion', 'text']]